*0.3 Classical NLP*

# Text classification with sklearn

**The situation.** 30,000 tickets a month need routing to billing, technical or account teams. An LLM call per ticket costs money and 800 ms. A junior engineer suggests "just use the LLM for everything". A classical classifier does this job in 2 ms per ticket on a CPU, and you can measure exactly how good it is.

**The classical pipeline.** TF-IDF features → logistic regression. Train on labelled examples, evaluate on held-out ones with a per-class report, save the model. Here on real data: 20 Newsgroups posts in three topics, which behave like three ticket queues.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

In [2]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.pipeline import make_pipeline

topics = ["sci.med", "sci.space", "rec.autos"]
train = fetch_20newsgroups(
    subset="train", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)
test = fetch_20newsgroups(
    subset="test", categories=topics, remove=("headers", "footers", "quotes"), random_state=0
)

model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2),
    LogisticRegression(C=10, max_iter=3000),
)
model.fit(train.data, train.target)
predictions = model.predict(test.data)
print(classification_report(test.target, predictions, target_names=train.target_names))
accuracy = model.score(test.data, test.target)
assert accuracy > 0.8

              precision    recall  f1-score   support

   rec.autos       0.80      0.91      0.85       396
     sci.med       0.90      0.84      0.87       396
   sci.space       0.90      0.84      0.87       394

    accuracy                           0.86      1186
   macro avg       0.87      0.86      0.86      1186
weighted avg       0.87      0.86      0.86      1186



**Reading the output.** Precision (of the posts routed to a queue, how many belonged there), recall (of the posts that belonged, how many were routed there) and F1 per class, plus the support (how many test posts). Around 85–90% on posts with headers and quotes stripped — text as messy as tickets.

**Speed, confidence, and saving the model.** The things production needs beyond accuracy.

In [3]:
import pickle
import time

import numpy as np

texts = test.data[:1000]
started = time.perf_counter()
probabilities = model.predict_proba(texts)
per_ticket_ms = (time.perf_counter() - started) / len(texts) * 1000
print(f"{per_ticket_ms:.2f} ms per ticket on CPU")

confident = probabilities.max(axis=1) >= 0.8
print(
    
        f"{confident.mean():.0%} of tickets routed automatically (≥80% confidence); the rest go to a "
        f"person"
    
)
predicted = model.classes_[probabilities.argmax(axis=1)]
actual = np.array(test.target[:1000])
print(
    "accuracy on the confident ones:", f"{np.mean(predicted[confident] == actual[confident]):.1%}"
)

saved = pickle.dumps(model)
print("saved model size:", round(len(saved) / 1e6, 1), "MB")
assert per_ticket_ms < 50

0.08 ms per ticket on CPU
39% of tickets routed automatically (≥80% confidence); the rest go to a person
accuracy on the confident ones: 100.0%
saved model size: 2.0 MB


**Reading the output.** A couple of milliseconds per ticket, and the confidence threshold splits the work: the confident majority is routed automatically at higher accuracy than the overall number; the uncertain rest goes to a person (or to an LLM). The saved model is a few megabytes.

**The rule to remember.** For a fixed set of categories with labelled data, a TF-IDF + logistic-regression pipeline is the baseline to beat: measurable, fast, cheap. Reach for an LLM when categories change often or labels do not exist.

| Use it when | Don't when | Instead use |
|---|---|---|
| fixed categories, ≥ a few hundred labelled examples, high volume | no labels; categories change weekly; needs reasoning over the text | LLM zero-shot classification; or use the LLM to label data, then train this |

**Watch out**
- Report per-class numbers, not one accuracy; a rare but important class can be terrible while the average looks fine.
- Retrain on a schedule; the words customers use drift.
- Pickle only models you trained; never unpickle files from outside.